In [2]:
import re
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse

## Part I - Import and reorganization to homogene h5ad structure

In [3]:
tcga = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/TCGA/TCGA_cancer_data.h5ad")

In [11]:
tcga.var_names = tcga.var["ensg_id"].copy()

In [15]:
tcga.write_h5ad("/cluster/work/boeva/eheiss/datasets/TCGA/tcga.h5ad")

## Part II - Statistics

In [3]:
with open("/cluster/work/boeva/eheiss/scbFM/data/gene_list.txt") as f:
    gene_list = [line.strip() for line in f if line.strip()]

In [7]:
tcga = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/TCGA/tcga.h5ad")
print(tcga )

AnnData object with n_obs × n_vars = 9784 × 22777
    obs: 'sample_id', 'patient_id', 'project', 'age_at_initial_pathologic_diagnosis', 'gender', 'race', 'ajcc_pathologic_tumor_stage', 'clinical_stage', 'histological_type', 'histological_grade', 'initial_pathologic_dx_year', 'menopause_status', 'birth_days_to', 'vital_status', 'tumor_status', 'last_contact_days_to', 'death_days_to', 'cause_of_death', 'new_tumor_event_type', 'new_tumor_event_site', 'new_tumor_event_site_other', 'new_tumor_event_dx_days_to', 'treatment_outcome_first_course', 'margin_status', 'residual_tumor', 'OS', 'OS.time', 'DSS', 'DSS.time', 'DFI', 'DFI.time', 'PFI', 'PFI.time', 'Redaction', 'tumor_type_id', 'label_idx'
    var: 'ensg_id'


In [8]:
gene_set = set(map(str, gene_list))
var_names = np.asarray(tcga.var_names.astype(str))

in_list_mask = np.array([g in gene_set for g in var_names], dtype=bool)
not_in_list_mask = ~in_list_mask
not_in_list_weights = not_in_list_mask.astype(np.float64)

chunk_size = 1000

sum_frac_nonzero = 0.0
sum_frac_reads = 0.0
n_obs_done = 0

for start in range(0, tcga.n_obs, chunk_size):
    end = min(start + chunk_size, tcga.n_obs)

    X_chunk = tcga.X[start:end]

    if sparse.issparse(X_chunk):
        X_chunk = X_chunk.tocsr()

        total_nonzero = np.asarray(X_chunk.getnnz(axis=1)).ravel()
        total_reads = np.asarray(X_chunk.sum(axis=1)).ravel()

        # Same as X_chunk[:, not_in_list_mask].sum(axis=1), but avoids sparse fancy indexing.
        not_in_list_reads = np.asarray(X_chunk @ not_in_list_weights).ravel()

        X_binary = X_chunk.copy()
        X_binary.data = np.ones_like(X_binary.data, dtype=np.float64)
        not_in_list_nonzero = np.asarray(X_binary @ not_in_list_weights).ravel()

        del X_binary

    else:
        X_chunk = np.asarray(X_chunk)

        total_nonzero = (X_chunk > 0).sum(axis=1)
        not_in_list_nonzero = (X_chunk[:, not_in_list_mask] > 0).sum(axis=1)

        total_reads = X_chunk.sum(axis=1)
        not_in_list_reads = X_chunk[:, not_in_list_mask].sum(axis=1)

    frac_nonzero_not_in_list = np.divide(
        not_in_list_nonzero,
        total_nonzero,
        out=np.zeros_like(total_nonzero, dtype=float),
        where=total_nonzero > 0,
    )

    frac_reads_not_in_list = np.divide(
        not_in_list_reads,
        total_reads,
        out=np.zeros_like(total_reads, dtype=float),
        where=total_reads > 0,
    )

    sum_frac_nonzero += frac_nonzero_not_in_list.sum()
    sum_frac_reads += frac_reads_not_in_list.sum()
    n_obs_done += end - start

    if start == 0 or n_obs_done % (10 * chunk_size) == 0 or end == tcga.n_obs:
        print(f"Processed {n_obs_done:,}/{tcga.n_obs:,} samples")

    del X_chunk

print("Average portion of non-zero genes NOT in gene_list:",
      sum_frac_nonzero / n_obs_done)

print("Average portion of total reads NOT in gene_list:",
      sum_frac_reads / n_obs_done)


Processed 1,000/9,784 samples
Processed 9,784/9,784 samples
Average portion of non-zero genes NOT in gene_list: 0.2220541544535524
Average portion of total reads NOT in gene_list: 0.15416850092337123


## Part III - filter to gene list

In [9]:
gene_list = [str(g) for g in gene_list]
gene_index = pd.Index(tcga.var_names.astype(str))

reorder_idx = gene_index.get_indexer(gene_list)
missing = [g for g, i in zip(gene_list, reorder_idx) if i < 0]
if missing:
    raise ValueError(f"{len(missing)} genes from gene_list are missing in tcga. First 20: {missing[:20]}")

out_path = "/cluster/work/boeva/eheiss/datasets/TCGA/tcga.h5ad"
chunk_size = 1000

chunks = []
    
for start in range(0, tcga.n_obs, chunk_size):
    end = min(start + chunk_size, tcga.n_obs)
    X_chunk = tcga.X[start:end, :]

    if sparse.issparse(X_chunk):
        # CSC handles column selection more reliably than CSR on some SciPy builds.
        X_chunk = X_chunk.tocsc()[:, reorder_idx].tocsr()
    else:
        X_chunk = np.asarray(X_chunk)[:, reorder_idx]

    chunk = ad.AnnData(
        X=X_chunk,
        obs=tcga.obs.iloc[start:end].copy(),
        var=pd.DataFrame(index=pd.Index(gene_list, name=tcga.var_names.name)),
    )
    chunks.append(chunk)

    print(f"Prepared {end:,}/{tcga.n_obs:,} samples")

tcga_aligned = ad.concat(chunks, axis=0, join="inner", merge="same")
tcga_aligned.var_names = pd.Index(gene_list, name=tcga.var_names.name)

tcga_aligned.write_h5ad(out_path)
print("Wrote:", out_path)
print("Shape:", tcga_aligned.shape)

Prepared 1,000/9,784 samples
Prepared 2,000/9,784 samples
Prepared 3,000/9,784 samples
Prepared 4,000/9,784 samples
Prepared 5,000/9,784 samples
Prepared 6,000/9,784 samples
Prepared 7,000/9,784 samples
Prepared 8,000/9,784 samples
Prepared 9,000/9,784 samples
Prepared 9,784/9,784 samples
Wrote: /cluster/work/boeva/eheiss/datasets/TCGA/tcga.h5ad
Shape: (9784, 13004)
